# Converting hifi to fastq

In [ ]:
(base) kfloer_smith_edu@umd-cscdr-cpu045:/work/pi_lmangiamele_smith_edu/yale_hifi_03_26/hifi_reads$ samtools fastq \
m84189_260311_040439_s4.hifi_reads.bam \
| gzip > sparvus_hifi.fastq.gz
[M::bam2fq_mainloop] discarded 0 singletons
[M::bam2fq_mainloop] processed 5933111 reads

# Hifiasm

## making new scratch directory

In [ ]:
mkdir sparvus_hifiasm
cd /scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm

## simlink to fastq

In [ ]:
(base) kfloer_smith_edu@login1:/scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm$ ln -s /work/pi_lmangiamele_smith_edu/yale_hifi_03_26/hifi_reads/sparvus_hifi.fastq.gz .
(base) kfloer_smith_edu@login1:/scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm$ ls
sparvus_hifi.fastq.gz

## create new envirnement

In [ ]:
mamba create -n assembly_env -c bioconda -c conda-forge hifiasm seqkit quast minimap2 -y

## config file

In [ ]:
nano sparvus_config.sh

In [ ]:
#!/bin/bash

# =========================
# Project directories
# =========================

SCRATCH_DIR=/scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm
PROJ_DIR=/home/kfloer_smith_edu/assembly
READS_DIR=/work/pi_lmangiamele_smith_edu/yale_hifi_03_26/hifi_reads
OUTPUT_DIR=${SCRATCH_DIR}/assembly_output

LOG_DIR=${PROJ_DIR}/job_logs

# =========================
# Input files
# =========================

HIFI_READS=${READS_DIR}/sparvus_hifi.fastq.gz

# =========================
# Software / environments
# =========================

CONDA_SH=/modules/opt/linux-ubuntu24.04-x86_64/miniforge3/24.7.1/etc/profile.d/conda.sh

CONDA_ENV=assembly_env

# =========================
# Assembly parameters
# =========================

THREADS=32
MEMORY=256G

ASSEMBLY_PREFIX=sparvus_hifiasm

## script for assembly

In [ ]:
#!/bin/bash
#SBATCH --job-name=sparvus_hifiasm
#SBATCH --nodes=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH --time=24:00:00
#SBATCH --output=/home/kfloer_smith_edu/assembly/job_logs/%x.%j.out
#SBATCH --error=/home/kfloer_smith_edu/assembly/job_logs/%x.%j.err

set -euo pipefail

# Load config
source /home/kfloer_smith_edu/assembly/scripts/sparvus_config.sh

# Create directories
mkdir -p "$OUTPUT_DIR"
mkdir -p "$LOG_DIR"

# Safety checks
[[ -f "$CONDA_SH" ]] || {
    echo "ERROR: conda.sh not found"
    exit 1
}

[[ -f "$HIFI_READS" ]] || {
    echo "ERROR: HiFi reads not found"
    exit 1
}

# Activate conda
source "$CONDA_SH"
conda activate "$CONDA_ENV"

# Check hifiasm
command -v hifiasm >/dev/null 2>&1 || {
    echo "ERROR: hifiasm not found"
    exit 1
}

echo "Using hifiasm:"
which hifiasm
hifiasm --version || true

echo "Starting assembly"
date

cd "$OUTPUT_DIR"

hifiasm \
    -o "$ASSEMBLY_PREFIX" \
    -t "$THREADS" \
    --primary \
    "$HIFI_READS" \
    2> "${ASSEMBLY_PREFIX}.log"

echo "Done"
date


In [ ]:
awk '/^S/{print ">"$2"\n"$3}' sparvus_hifiasm.p_ctg.gfa > sparvus_primary_contigs.fa

In [ ]:
seqkit stats sparvus_primary_contigs.fa > sparvus_primary_contigs.seqkit_stats.txt

seqkit stats -a sparvus_primary_contigs.fa > sparvus_primary_contigs.seqkit_stats_full.txt

In [ ]:
(base) kfloer_smith_edu@login1:/scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm/assembly_output$ head -n 3 sparvus_primary_contigs.fa | cut -c1-40
>ptg000001l
ATAATCATGTATACATTAATGTACTATTCAGTGTTGTTAT
>ptg000002l
(base) kfloer_smith_edu@login1:/scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm/assembly_output$ head -n 3 sparvus_hifiasm.p_ctg.gfa  | cut -c1-40
S	ptg000001l	ATAATCATGTATACATTAATGTACTAT
A	ptg000001l	0	+	m84189_260311_040439_s4
A	ptg000001l	3474	+	m84189_260311_040439


In [ ]:
sparvus_hifiasm/assembly_output$ cat sparvus_primary_contigs.seqkit_stats.txt

Assembly size: 4.48 Gb
Contigs:      4,082
Avg length:   ~1.1 Mb
Longest:      36.8 Mb

In [ ]:
sparvus_hifiasm/assembly_output$ cat sparvus_primary_contigs.seqkit_stats_full.txt

Assembly size: 4.48 Gb
Contigs: 4,082
Contig N50: 6.88 Mb
Longest contig: 36.84 Mb
GC: 43.07%
Gaps/Ns: 0



In [ ]:
#!/bin/bash
#SBATCH --job-name=sparvus_busco
#SBATCH --nodes=1
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH --time=24:00:00
#SBATCH --output=/home/kfloer_smith_edu/assembly/job_logs/%x.%j.out
#SBATCH --error=/home/kfloer_smith_edu/assembly/job_logs/%x.%j.err

set -euo pipefail

# =========================
# Load config
# =========================

source /home/kfloer_smith_edu/assembly/scripts/sparvus_config.sh

# =========================
# BUSCO settings
# =========================

BUSCO_OUT=busco_primary_vertebrata

ASSEMBLY_FA=${OUTPUT_DIR}/sparvus_primary_contigs.fa

LINEAGE=vertebrata_odb10

# =========================
# Safety checks
# =========================

[[ -f "$CONDA_SH" ]] || {
    echo "ERROR: conda.sh not found"
    exit 1
}

[[ -f "$ASSEMBLY_FA" ]] || {
    echo "ERROR: Assembly FASTA not found"
    exit 1
}

# =========================
# Activate BUSCO environment
# =========================

source "$CONDA_SH"

conda activate busco_env

# =========================
# Tool checks
# =========================

command -v busco >/dev/null 2>&1 || {
    echo "ERROR: BUSCO not found"
    exit 1
}

echo "Using BUSCO:"
which busco

echo "Starting BUSCO"
date

cd "$OUTPUT_DIR"

# =========================
# Run BUSCO
# =========================

busco \
    -i "$ASSEMBLY_FA" \
    -l "$LINEAGE" \
    -m genome \
    -o "$BUSCO_OUT" \
    -c 16

echo "BUSCO finished"
date


In [ ]:
tail -f /home/kfloer_smith_edu/assembly/job_logs/sparvus_busco.*.out

In [ ]:
ls -lh /scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm/assembly_output/busco_primary_vertebrata

# Contamination check

In [ ]:
cd /scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm/contamination_check

module load apptainer/latest
git clone https://github.com/ncbi/fcs.git
cd fcs

In [ ]:
export FCS_USE_DOCKER=0

In [ ]:
mkdir -p ~/bin
cat > ~/bin/singularity <<'EOF'
#!/bin/bash
exec apptainer "$@"
EOF

chmod +x ~/bin/singularity
export PATH=~/bin:$PATH

which singularity
singularity --version

In [ ]:
cd /scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm/contamination_check/fcs

export APPTAINER_CACHEDIR=/work/pi_lmangiamele_smith_edu/.apptainer/cache
mkdir -p "$APPTAINER_CACHEDIR"

apptainer pull fcs-adaptor.sif docker://ncbi/fcs-adaptor:latest

In [ ]:
mkdir -p ../fcs_adaptor_out

time bash dist/run_fcsadaptor.sh \
  --fasta-input /scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm/assembly_output/sparvus_primary_contigs.fa \
  --output-dir ../fcs_adaptor_out \
  --container-engine singularity \
  --image fcs-adaptor.sif \
  --euk

In [ ]:
ls -lh ../fcs_adaptor_out
cat ../fcs_adaptor_out/*report* | head -80

In [ ]:
# ran script saved on github

# Genome Contamination

In [ ]:
cd /scratch3/workspace/kfloer_smith_edu-simple/sparvus_hifiasm/contamination_check/fcs

module load apptainer/latest
export PATH=/home/kfloer_smith_edu/bin:$PATH
export APPTAINER_CACHEDIR=/work/pi_lmangiamele_smith_edu/.apptainer/cache
mkdir -p "$APPTAINER_CACHEDIR"

mkdir -p ../fcs_gx_db ../fcs_gx_out

In [ ]:
apptainer pull fcs-gx.sif docker://ncbi/fcs-gx:latest

In [ ]:
python3 dist/fcs.py --image fcs-gx.sif db get --help